This MWE will perform QSE for the ethylene molecule with an active space of four electrons in four spatial orbitals. First construct the HF state and qubit operator for the system.

In [ ]:
from pyscf import scf, gto
from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.pyscf import active_space_from_mf

geometry = [["C",   [0.00000,   0.00000,   0.66730]],
            ["C",   [0.00000,   0.00000,  -0.66730]],
            ["H",   [0.00000,   0.92395,   1.24109]],
            ["H",   [0.00000,  -0.92395,   1.24109]],
            ["H",   [0.00000,  -0.92395,  -1.24109]],
            ["H",   [0.00000,   0.92395,  -1.24109]]]

mol = gto.M(atom=geometry, basis="sto3g")
mol.build()
mf = scf.RHF(mol)
mf.kernel()
integrals, onv = active_space_from_mf(mf, 4, 4)
qop = JordanWigner().encode_operator(restricted_integrals_to_fermion_operator(*integrals))
n_qubits = 8

We now construct an approximate ground state by performing VQE with a UCCSD ansatz.

In [ ]:
from qarp.blocks import UCCBlock, MappedONVStateBlock, CompositeBlock
from qarp.algorithms import VQE
import numpy as np

ansatz = CompositeBlock(
    [MappedONVStateBlock(onv, JordanWigner()), UCCBlock(onv)],
)
ansatz.build()
params=np.array([0.] * len(ansatz.symbols))
# params=None
vqe = VQE(qop, ket=ansatz, initial_parameters=params, verbose=True, gradient=True)
vqe.build()
e_vqe, p_vqe = vqe.run()

We now construct and call the QSE object.

In [ ]:
from qarp.algorithms import QSE, StateVector
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.engines import QarpEngine

ground_state = vqe.final_block

excitations = JordanWigner().encode_operator(ucc_singles_and_doubles(onv, generalised=False, antihermitized=True)[0])

qse = QSE(
    hamiltonian = qop,
    ground_state = ground_state, # Approx ground state from VQE. Cannot be symbolic.
    primitive = StateVector(), # primitive for evaluating the QSE Hamiltonian matrix
    overlap_primitive = StateVector(), # primitive to evaluating the QSE Overlap matrix
    excitation_operators=excitations, # Qubit operators for constructing the excitations A_j.
    engine = QarpEngine(),
    real_symmetric=False, # here we do not assume real and symmetric H and S matrices. Instead assumed Hermitian.
    verbose=True
)
qse.build()
w, v = qse.run()

Let's compare with the exact eigenvalues for the correct particle number with those we found using QSE.

In [ ]:
from qarp.operators import FermionOperator
import matplotlib.pyplot as plt
import numpy as np

fullw, fullv = np.linalg.eigh(qop.sparse_matrix(n_qubits).todense())
particle_number_operator = FermionOperator()
for i in range(n_qubits):
    particle_number_operator += FermionOperator(f"{i}^ {i}")

qpno = JordanWigner().encode_operator(particle_number_operator).sparse_matrix(n_qubits).todense()
occs = (fullv.T @ qpno @ fullv).real

pno_conserving_eigs = []
for i in range(len(occs)):
    if abs(occs[i, i] - sum(onv)) < 1e-10:
        pno_conserving_eigs.append(fullw[i])
energy_thresh = -75.4
pno_conserving_eigs = [eig.real for eig in pno_conserving_eigs if eig < energy_thresh]

qse_eig_indices = []
for eig in w:
    index = min(range(len(pno_conserving_eigs)), key=lambda i: abs(pno_conserving_eigs[i]-eig))
    qse_eig_indices += [index]

plt.plot(list(range(len(pno_conserving_eigs))), pno_conserving_eigs, label="n-particle eigs", marker="o", linestyle="None", color="black")
plt.plot(qse_eig_indices, w.real, label="QSE approx eigs", marker="X", linestyle="None", color="red")
plt.plot([0], [e_vqe.real], label="VQE solution", marker="X", linestyle="None", color="blue")
plt.xlabel("Index")
plt.ylabel("Energy")
plt.legend()
plt.grid(True)
plt.plot()